# Minimum Detectable Altitude for Horizon Curvature

**Question**: At what altitude does Earth's curvature become detectable in a photograph?

**Approach**: Compute the vertical span of the horizon arc across the image. When this span exceeds 1 pixel, curvature is detectable.

This notebook demonstrates the calculation and explores how different cameras and altitudes affect detectability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from planet_ruler.geometry import limb_arc
from planet_ruler.camera import CAMERA_DB
from IPython.display import Image

# Earth parameters
R_EARTH = 6.371e6  # meters

print("✓ Imports complete")

## The Sagitta: Measuring Curvature

The **sagitta** (from Latin "arrow") is the perpendicular distance from a chord to its arc:

If we imagine a roughly centered image of the horizon, we can measure the **vertical span** of the limb across the image width:

$$s= |y_{\text{max}} - y_{\text{min}}|$$

When $s < 1$ pixel, the horizon (`Arc` below) appears as a straight line.

In [ ]:
Image(filename="../demo/images/sagitta_diagram.png") 

## Computing Limb Span

Using Planet Ruler's `limb_arc()` function, we compute the exact limb position at the image edges.

In [ ]:
def compute_limb_span(altitude_m, focal_length_mm, sensor_width_mm, 
                      image_width_px, image_height_px, planet_radius_m=R_EARTH):
    """
    Compute vertical span of limb across image.
    
    Args:
        altitude_m: Altitude above surface (m)
        focal_length_mm: Camera focal length (mm)
        sensor_width_mm: Sensor width (mm)
        image_width_px: Image width (pixels)
        image_height_px: Image height (pixels)
        planet_radius_m: Planet radius (m)
    
    Returns:
        float: Vertical span in pixels (y_right - y_left)
    """
    # Convert to meters
    f_m = focal_length_mm / 1000.0
    w_m = sensor_width_mm / 1000.0
    
    # Compute limb at leftmost and rightmost pixels
    x_edges = np.array([0, image_width_px - 1])
    
    y_limb = limb_arc(
        r=planet_radius_m,
        n_pix_x=image_width_px,
        n_pix_y=image_height_px,
        h=altitude_m,
        f=f_m,
        w=w_m,
        x0=0, y0=0,
        theta_x=limb_camera_angle(planet_radius_m, altitude_m),
        theta_y=0,
        theta_z=0,
        x_coords=x_edges
        # x_coords=np.arange(image_width_px),
    )
    
    # plt.plot(np.arange(image_width_px), y_limb)
    # plt.show()

    # Return vertical span
    return np.ptp(y_limb)

# Test with a phone camera
span_10m = compute_limb_span(
    altitude_m=10,
    focal_length_mm=5.4,  # Samsung S22+ main
    sensor_width_mm=7.6,
    image_width_px=4000,
    image_height_px=3000
)

print(f"Limb span at 10m altitude: {span_10m:.2f} pixels")
print(f"Detectable? {span_10m >= 1.0}")

## Altitude Scan: How Span Varies with Height

Let's scan across altitudes from 1 meter to 100 km and see how the detectable curvature changes.

In [ ]:
# Altitude range: 1 m to 100 km
altitudes = np.logspace(0, 5, 100)  # log scale from 10^0 to 10^5 meters

# Camera: Samsung S22+ main camera
focal_length = 5.4  # mm
sensor_width = 7.6  # mm
image_width = 4000
image_height = 3000

# Compute span at each altitude
spans = []
for h in altitudes:
    try:
        span = compute_limb_span(h, focal_length, sensor_width, 
                                image_width, image_height)
        spans.append(span)
    except:
        spans.append(0)

spans = np.array(spans)

# Plot
plt.figure(figsize=(12, 6))
plt.loglog(altitudes, spans, linewidth=2, label='Samsung S22+ Main')
plt.axhline(y=1, color='r', linestyle='--', linewidth=2, 
            label='Theoretical Detection threshold (1 pixel)')

# Mark reference altitudes
refs = {
    'Ground': 1,
    'Airplane': 10e3,
    'Weather balloon': 30e3,
    'Space (Kármán)': 100e3
}
for name, alt in refs.items():
    idx = np.argmin(np.abs(altitudes - alt))
    plt.plot(alt, spans[idx], 'o', markersize=8)
    plt.annotate(f'{name}\n{spans[idx]:.0f} px', 
                xy=(alt, spans[idx]),
                xytext=(10, 10), textcoords='offset points',
                fontsize=9, alpha=0.8)

plt.xlabel('Altitude (m)', fontsize=12)
plt.ylabel('Limb Vertical Span (pixels)', fontsize=12)
plt.title('Horizon Curvature Detectability vs Altitude\nSamsung S22+ Main Camera (4000×3000 px)', 
         fontsize=14)
plt.grid(True, alpha=0.3, which='both')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nMinimum altitude (span ≥ 1 px): {altitudes[spans >= 1][0]:.1f} m")

## Camera Comparison: Wide vs Telephoto

How do different focal lengths affect minimum altitude?

In [ ]:
# Define cameras to compare
cameras = [
    {'name': 'Ultrawide (1.5mm)', 'f': 1.5, 'w': 5.7, 'W': 4000, 'H': 3000},
    {'name': 'Wide (5.4mm)', 'f': 5.4, 'w': 7.6, 'W': 4000, 'H': 3000},
    {'name': 'Telephoto 3× (9mm)', 'f': 9.0, 'w': 5.7, 'W': 4000, 'H': 3000},
    {'name': 'Telephoto 10× (23mm)', 'f': 23.0, 'w': 4.3, 'W': 3280, 'H': 2460},
]

plt.figure(figsize=(12, 7))

for cam in cameras:
    spans = []
    for h in altitudes:
        try:
            span = compute_limb_span(h, cam['f'], cam['w'], 
                                    cam['W'], cam['H'])
            spans.append(span)
        except:
            spans.append(0)
    
    spans = np.array(spans)
    plt.loglog(altitudes, spans, linewidth=2, label=cam['name'])

plt.axhline(y=1, color='r', linestyle='--', linewidth=2, 
            label='Detection threshold')

plt.xlabel('Altitude (m)', fontsize=12)
plt.ylabel('Limb Vertical Span (pixels)', fontsize=12)
plt.title('Camera Type Comparison: Curvature Detection vs Altitude', fontsize=14)
plt.grid(True, alpha=0.3, which='both')
plt.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()

## Finding Minimum Altitude: Binary Search

Let's find the exact minimum altitude for each camera.

In [ ]:
def find_minimum_altitude(focal_length_mm, sensor_width_mm, 
                         image_width_px, image_height_px,
                         planet_radius_m=R_EARTH):
    """
    Binary search to find altitude where span = 1 pixel.
    
    Returns:
        float: Minimum altitude in meters
    """
    h_low = 0.1
    h_high = 1e6
    tolerance = 0.1
    
    for _ in range(50):
        h_mid = (h_low + h_high) / 2
        
        span = compute_limb_span(h_mid, focal_length_mm, sensor_width_mm,
                                image_width_px, image_height_px, planet_radius_m)
        
        if span < 1.0:
            h_low = h_mid
        else:
            h_high = h_mid
        
        if h_high - h_low < tolerance:
            break
    
    return h_high

# Calculate for each camera
print("Minimum Detectable Altitude\n" + "="*50)
for cam in cameras:
    h_min = find_minimum_altitude(cam['f'], cam['w'], cam['W'], cam['H'])
    print(f"{cam['name']:<25} {h_min:>8.1f} m")

## Error Budget Visualization

At each altitude, the geometric signal (limb span) provides a yardstick for acceptable measurement error.

For accurate radius measurement, fitting errors should be much smaller than the geometric signal.

In [ ]:
# Typical measurement error levels
error_levels = {
    'Sub-pixel fitting': 0.5,
    'Pixel-level noise': 2.0,
    'Atmospheric distortion': 10.0,
    'Poor fit': 50.0
}

# Compute spans for wide camera
cam_wide = cameras[1]  # 5.4mm wide
spans_wide = []
for h in altitudes:
    try:
        span = compute_limb_span(h, cam_wide['f'], cam_wide['w'], 
                                cam_wide['W'], cam_wide['H'])
        spans_wide.append(span)
    except:
        spans_wide.append(0)

spans_wide = np.array(spans_wide)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Signal vs error
ax1.loglog(altitudes, spans_wide, linewidth=3, label='Geometric signal', color='blue')
for name, level in error_levels.items():
    ax1.axhline(y=level, linestyle='--', linewidth=1.5, alpha=0.7, label=name)

ax1.set_xlabel('Altitude (m)', fontsize=12)
ax1.set_ylabel('Pixels', fontsize=12)
ax1.set_title('Geometric Signal vs Measurement Errors', fontsize=13)
ax1.grid(True, alpha=0.3, which='both')
ax1.legend(fontsize=10)

# Right: Signal-to-noise ratio
snr_subpixel = spans_wide / 0.5
snr_pixel = spans_wide / 2.0
snr_atmospheric = spans_wide / 10.0

ax2.loglog(altitudes, snr_subpixel, linewidth=2, label='vs sub-pixel error')
ax2.loglog(altitudes, snr_pixel, linewidth=2, label='vs pixel-level noise')
ax2.loglog(altitudes, snr_atmospheric, linewidth=2, label='vs atmospheric error')
ax2.axhline(y=10, color='g', linestyle='--', linewidth=2, label='SNR=10 (good)')
ax2.axhline(y=3, color='orange', linestyle='--', linewidth=2, label='SNR=3 (marginal)')

ax2.set_xlabel('Altitude (m)', fontsize=12)
ax2.set_ylabel('Signal-to-Noise Ratio', fontsize=12)
ax2.set_title('Measurement Quality vs Altitude', fontsize=13)
ax2.grid(True, alpha=0.3, which='both')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Key Insights

1. **Curvature is detectable from ground level** (1-3 m) with modern phone cameras
2. **Telephoto lenses need higher altitudes** (up to 34 m for 10× zoom) due to narrower field of view
3. **Signal quality improves rapidly with altitude**:
   - At 10 m: ~4 pixels (marginal)
   - At 100 m: ~12 pixels (good)
   - At 10 km: ~115 pixels (excellent)
4. **Error budgets scale with altitude**: Higher altitudes tolerate more measurement error while maintaining accuracy

## Practical Implications

The limiting factors for horizon-based radius measurement are **not geometric** but rather:

- **Atmospheric effects**: Refraction, haze, turbulence
- **Scene complexity**: Buildings, trees, clouds obscuring horizon
- **Optical quality**: Lens aberrations, sensor noise
- **Fitting precision**: Algorithm accuracy in detecting the limb

These effects decrease with altitude, making high-altitude measurements (airplanes, balloons, space) both easier and more accurate than the theoretical minimum would suggest.